# IBM AML Iceberg Queries

This notebook is dedicated to Iceberg/Trino comparisons.

**Simple mode:** run the notebook cells; it will:
1. Start local Iceberg services
2. Download/use AML Small source data
3. Normalize to `aml-demo.csv` using `MAX_ROWS`
4. Seed Iceberg with live progress using the fast seeding script


## Optional: Run Local Iceberg Setup From Notebook

This cell can run the local setup scripts directly:
1. `bash ./scripts/iceberg_local_down.sh`
2. `bash ./scripts/iceberg_local_up.sh`

Seeding is handled automatically later based on `MAX_ROWS`.
`RUN_DOWN` defaults to `False` to avoid deleting volumes unless you opt in.


In [1]:
from pathlib import Path
import subprocess
from IPython.display import display, Markdown

REPO_ROOT = Path.home() / "SourceCode/graph-query-engine"
RUN_DOWN = False
RUN_UP = True
AUTO_RUN_SETUP = True


def run_local_iceberg_setup(run_down: bool = False, run_up: bool = True) -> bool:
    if not REPO_ROOT.exists():
        display(Markdown(f"**Error:** repo path not found: `{REPO_ROOT}`"))
        return False

    steps = []
    if run_down:
        steps.append("bash ./scripts/iceberg_local_down.sh")
    if run_up:
        steps.append("bash ./scripts/iceberg_local_up.sh")

    if not steps:
        display(Markdown("No setup steps selected. Set one of `RUN_DOWN` or `RUN_UP` to `True`."))
        return False

    display(Markdown(f"Running setup in `{REPO_ROOT}`"))
    ok = True
    for i, cmd in enumerate(steps, 1):
        display(Markdown(f"**Step {i}:** `{cmd}`"))
        proc = subprocess.Popen(
            cmd,
            cwd=REPO_ROOT,
            shell=True,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
        rc = proc.wait()
        if rc == 0:
            display(Markdown("- Status: OK"))
        else:
            display(Markdown(f"- Status: FAILED (exit {rc})"))
            ok = False
            break

    display(Markdown("**Setup completed successfully.**" if ok else "**Setup failed.**"))
    return ok


if AUTO_RUN_SETUP:
    run_local_iceberg_setup(run_down=RUN_DOWN, run_up=RUN_UP)
else:
    print("Set AUTO_RUN_SETUP=True and run this cell, or call run_local_iceberg_setup(...)")


Running setup in `/Users/vjoshi/SourceCode/graph-query-engine`

**Step 1:** `bash ./scripts/iceberg_local_up.sh`

 Container iceberg-minio  Running
 Container iceberg-rest  Running
 Container iceberg-trino  Running
 Container iceberg-minio  Waiting
 Container iceberg-minio  Healthy
 Container iceberg-minio-init  Starting
 Container iceberg-minio-init  Started
 Container iceberg-minio-init  Waiting
 Container iceberg-minio  Waiting
 Container iceberg-minio-init  Exited
 Container iceberg-minio  Healthy
Waiting for Trino to accept queries...
Iceberg local stack is ready.


- Status: OK

**Setup completed successfully.**

In [2]:
import requests
import pandas as pd
import subprocess
import json as _json
from typing import Dict, Any
from IPython.display import display, Markdown
from pathlib import Path

BASE_URL = "http://localhost:7000"
ICEBERG_MAPPING_PATH = str(Path.cwd() / "mappings/iceberg-local-mapping.json")
ICEBERG_MAPPING_ID = "iceberg-local"
MAX_ROWS = 1000000
TRINO_CONTAINER = "iceberg-trino"
TRINO_SERVER = "http://localhost:8080"
TRINO_CATALOG = "iceberg"
TRINO_SCHEMA = "aml"
SHOW_PLAN = False
FORCE_RESEED = False
FILE_SOURCE_FORMAT = "csv"


In [3]:
def get_sql_explain(gremlin_query: str) -> Dict[str, Any]:
    try:
        response = requests.post(
            f"{BASE_URL}/query/explain",
            json={"gremlin": gremlin_query},
            headers={"Content-Type": "application/json"},
            timeout=10,
        )
        if response.ok:
            return response.json()
        return {"error": f"HTTP {response.status_code}: {response.text}"}
    except Exception as e:
        return {"error": str(e)}


def upload_iceberg_mapping() -> bool:
    try:
        with open(ICEBERG_MAPPING_PATH, "rb") as f:
            resp = requests.post(
                f"{BASE_URL}/mapping/upload",
                params={"id": ICEBERG_MAPPING_ID, "name": "Iceberg Local", "activate": "false"},
                files={"file": ("iceberg-local-mapping.json", f, "application/json")},
                timeout=15,
            )
        m = resp.json()
        if "error" not in m:
            display(Markdown(f"Iceberg mapping ready - id=`{m.get('mappingId')}`"))
            return True
        display(Markdown(f"**Mapping upload failed:** {m['error']}"))
        return False
    except Exception as e:
        display(Markdown(f"**Mapping upload error:** {e}"))
        return False


def get_iceberg_sql(gremlin_query: str, include_plan: bool = False) -> tuple[str, list, str | None, dict | None]:
    try:
        params = {"plan": "true"} if include_plan else {}
        resp = requests.post(
            f"{BASE_URL}/query/explain",
            json={"gremlin": gremlin_query},
            headers={"Content-Type": "application/json", "X-Mapping-Id": ICEBERG_MAPPING_ID},
            params=params,
            timeout=10,
        )
        if resp.ok:
            data = resp.json()
            return data.get("translatedSql", ""), data.get("parameters", []), None, data.get("plan")
        err = None
        try:
            err = resp.json().get("error")
        except Exception:
            err = resp.text
        return "", [], f"HTTP {resp.status_code}: {err}", None
    except Exception as e:
        return "", [], str(e), None


def _sql_literal(value) -> str:
    if value is None:
        return "NULL"
    if isinstance(value, bool):
        return "TRUE" if value else "FALSE"
    if isinstance(value, (int, float)):
        return str(value)
    s = str(value).replace("'", "''")
    return f"'{s}'"


def bind_sql_parameters(sql: str, params: list) -> str:
    out = sql
    for p in params:
        if "?" not in out:
            raise ValueError("More SQL parameters were supplied than placeholders")
        out = out.replace("?", _sql_literal(p), 1)
    if "?" in out:
        raise ValueError("SQL still contains unbound placeholders")
    return out


def run_trino(sql: str) -> pd.DataFrame:
    result = subprocess.run(
        ["docker", "exec", "-i", TRINO_CONTAINER, "trino", "--server", TRINO_SERVER, "--catalog", TRINO_CATALOG, "--schema", TRINO_SCHEMA, "--output-format", "JSON", "--execute", sql],
        capture_output=True,
        text=True,
        timeout=30,
    )
    if result.returncode != 0:
        stderr_lines = [ln.strip() for ln in (result.stderr or "").splitlines() if ln.strip()]
        useful = [ln for ln in stderr_lines if "org.jline.utils.Log" not in ln]
        msg = (useful[-1] if useful else (stderr_lines[-1] if stderr_lines else "Trino execution failed"))
        return pd.DataFrame([{"error": msg[:500]}])
    rows = [_json.loads(l) for l in result.stdout.strip().splitlines() if l.strip()]
    return pd.DataFrame(rows if rows else [{"result": "empty"}])


def resolve_raw_source_csv() -> str | None:
    candidates = [
        REPO_ROOT / "demo/data/HI-Small_Trans.csv",
        REPO_ROOT / "data/HI-Small_Trans.csv",
    ]
    for p in candidates:
        if p.exists():
            return str(p)
    demo_data = REPO_ROOT / "demo/data"
    if demo_data.exists():
        matches = sorted(demo_data.glob("HI-*_Trans.csv"))
        if matches:
            return str(matches[0])
    return None


def normalized_csv_path() -> str:
    return str(REPO_ROOT / "demo/data/aml-demo.csv")


def count_csv_rows(csv_path: str, max_rows: int) -> int:
    import csv
    rows = 0
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        next(reader, None)
        for _ in reader:
            rows += 1
            if rows >= max_rows:
                break
    return rows


def prepare_csv_for_max_rows(auto_download: bool = True, max_rows: int = 100000) -> str | None:
    raw_source = resolve_raw_source_csv()
    if not raw_source and auto_download:
        cmd = [
            "bash",
            "./scripts/download_aml_data.sh",
            "--variant",
            "HI-Small",
            "--rows",
            str(max_rows),
        ]
        display(Markdown(f"Downloading HI-Small dataset: `{ ' '.join(cmd) }`"))
        proc = subprocess.run(cmd, cwd=REPO_ROOT, text=True, capture_output=True)
        if proc.stdout:
            print(proc.stdout)
        if proc.stderr:
            print(proc.stderr)
        if proc.returncode == 0:
            raw_source = resolve_raw_source_csv()

    if not raw_source:
        return None

    out_csv = normalized_csv_path()
    normalize_cmd = [
        "python3",
        str(REPO_ROOT / "scripts/normalize_aml.py"),
        "--src",
        str(raw_source),
        "--dst",
        str(out_csv),
        "--rows",
        str(max_rows),
    ]
    display(Markdown(f"Preparing normalized CSV for MAX_ROWS: `{ ' '.join(normalize_cmd) }`"))
    proc = subprocess.run(normalize_cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode != 0:
        return None
    return out_csv if Path(out_csv).exists() else None


def get_iceberg_transfer_count() -> int | None:
    probe = run_trino("SELECT count(*) AS c FROM transfers")
    if "error" in probe.columns:
        return None
    if "c" not in probe.columns or len(probe.index) == 0:
        return 0
    return int(probe.iloc[0]["c"])


def preflight_trino_catalogs() -> bool:
    check = subprocess.run(
        ["docker", "exec", "-i", TRINO_CONTAINER, "trino", "--server", TRINO_SERVER, "--execute", "SHOW CATALOGS"],
        capture_output=True,
        text=True,
        timeout=20,
    )
    if check.returncode != 0:
        detail = (check.stderr or check.stdout or "Unable to query Trino catalogs").strip()
        display(Markdown(f"**Catalog preflight failed:** `{detail[:300]}`"))
        return False

    catalogs = {ln.strip().strip('"') for ln in (check.stdout or "").splitlines() if ln.strip()}
    has_iceberg = "iceberg" in catalogs
    has_hive = "hive" in catalogs

    display(Markdown(f"Catalog preflight: iceberg=`{has_iceberg}`, hive=`{has_hive}`"))

    if not has_iceberg:
        display(Markdown("**Required catalog missing:** `iceberg`"))
        return False

    if not has_hive:
        display(Markdown("**Required catalog missing:** `hive`"))
        display(Markdown("This notebook is configured for true file-based loading from object storage via Hive external tables."))
        return False

    return True


def run_iceberg_seed_script(csv_path: str, max_rows: int) -> bool:
    cmd = [
        "bash",
        "./scripts/iceberg_seed_trino_files.sh",
        "--csv",
        str(csv_path),
        "--rows",
        str(max_rows),
        "--format",
        FILE_SOURCE_FORMAT,
    ]
    display(Markdown(f"Seeding Iceberg tables with **{max_rows:,}** rows (true file-based `{FILE_SOURCE_FORMAT}` path via object storage)..."))
    display(Markdown("Real-time progress tracking with per-step timing:"))
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    return proc.wait() == 0


def ensure_iceberg_seed_ready(auto_download: bool = True, max_rows: int = 100000) -> None:
    csv_path = prepare_csv_for_max_rows(auto_download=auto_download, max_rows=max_rows)
    if not csv_path:
        display(Markdown("**Unable to prepare normalized CSV for Iceberg seeding.**"))
        return

    expected_rows = count_csv_rows(csv_path, max_rows)
    current_rows = get_iceberg_transfer_count()

    display(Markdown(f"CSV rows considered (up to MAX_ROWS): **{expected_rows}**"))
    display(Markdown(f"Current Iceberg transfers rows: **{current_rows if current_rows is not None else 'unavailable'}**"))

    if not FORCE_RESEED and current_rows is not None and current_rows == expected_rows and expected_rows > 0:
        display(Markdown(f"Iceberg data already loaded and up-to-date (`transfers={current_rows}`)."))
        return

    seeded = run_iceberg_seed_script(csv_path, max_rows)
    if not seeded:
        display(Markdown("**Iceberg seeding failed.** Check output above."))
        return

    refreshed_rows = get_iceberg_transfer_count()
    if expected_rows > 0 and (refreshed_rows is None or refreshed_rows == 0):
        display(Markdown("**Iceberg seeding appears to have failed (transfers row count is still 0).**"))
        display(Markdown("Check script output above for catalog/plugin errors and fallback behavior."))
        return
    display(Markdown(f"Iceberg seeding complete. Transfers rows: **{refreshed_rows if refreshed_rows is not None else 'unavailable'}**"))


def run_iceberg_query(gremlin_query: str, tx_mode: bool = False) -> Dict[str, Any]:
    # tx_mode is accepted for shape parity with aml_demo notebook, but Iceberg path ignores it.
    sql, params, err, plan = get_iceberg_sql(gremlin_query, include_plan=SHOW_PLAN)
    return {
        "gremlin": gremlin_query,
        "icebergSql": sql,
        "parameters": params,
        "plan": plan,
        "icebergError": err,
    }


def display_iceberg_result(gremlin: str, result: Dict[str, Any], title: str = "", limit: int = 10, tx_mode: bool = False):
    if title:
        display(Markdown(f"### {title}"))
    display(Markdown("**Gremlin:**"))
    display(Markdown(f"```groovy\n{gremlin}\n```"))

    sql = result.get("icebergSql", "")
    params = result.get("parameters", [])
    plan = result.get("plan")
    err = result.get("icebergError")

    if not sql:
        if err:
            display(Markdown(f"*Iceberg SQL translation not available: {err}*"))
        else:
            display(Markdown("*Iceberg SQL translation not available for this query.*"))
        return

    display(Markdown("**Iceberg SQL Translation:**"))
    display(Markdown(f"```sql\n{sql}\n```"))
    if params:
        display(Markdown(f"**Parameters:** `{params}`"))
    if plan:
        display(Markdown("**Query Plan:**"))
        display(Markdown(f"```json\n{_json.dumps(plan, indent=2)}\n```"))

    try:
        executable_sql = bind_sql_parameters(sql, params)
    except Exception as e:
        display(Markdown(f"*Failed to bind SQL parameters: {e}*"))
        return

    trino_df = run_trino(executable_sql)
    if "error" in trino_df.columns:
        display(Markdown(f"**Error:** {trino_df.iloc[0]['error']}"))
        return
    display(trino_df.head(limit))


upload_iceberg_mapping()
AUTO_DOWNLOAD_IF_MISSING = True
if preflight_trino_catalogs():
    ensure_iceberg_seed_ready(auto_download=AUTO_DOWNLOAD_IF_MISSING, max_rows=MAX_ROWS)
else:
    display(Markdown("Skipping seeding because catalog preflight did not pass."))


Iceberg mapping ready - id=`iceberg-local`

Catalog preflight: iceberg=`True`, hive=`True`

Preparing normalized CSV for MAX_ROWS: `python3 /Users/vjoshi/SourceCode/graph-query-engine/scripts/normalize_aml.py --src /Users/vjoshi/SourceCode/graph-query-engine/demo/data/HI-Small_Trans.csv --dst /Users/vjoshi/SourceCode/graph-query-engine/demo/data/aml-demo.csv --rows 1000000`

Reading from: /Users/vjoshi/SourceCode/graph-query-engine/demo/data/HI-Small_Trans.csv
Writing to:   /Users/vjoshi/SourceCode/graph-query-engine/demo/data/aml-demo.csv
Max rows:     1000000

Raw headers: ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

Done!
  Rows written:    1000000
  Suspicious rows: 575 (0.06%)
  Clean rows:      999425

Ready to load:
  curl -X POST "http://localhost:7000/admin/load-aml-csv?path=$(pwd)//Users/vjoshi/SourceCode/graph-query-engine/demo/data/aml-demo.csv&maxRows=1000000"



CSV rows considered (up to MAX_ROWS): **1000000**

Current Iceberg transfers rows: **200**

Seeding Iceberg tables with **1,000,000** rows (true file-based `csv` path via object storage)...

Real-time progress tracking with per-step timing:

🚀 True file-based Iceberg seeding via Hive external tables
Input CSV: /Users/vjoshi/SourceCode/graph-query-engine/demo/data/aml-demo.csv
Max rows: 1000000
File format: csv

[1/5] Generating per-table seed files...
[1/3] Parsing source CSV: /Users/vjoshi/SourceCode/graph-query-engine/demo/data/aml-demo.csv
[2/3] Writing intermediate CSV files to /tmp/iceberg-csv-gen
  ✓ countries: 10 rows → countries.csv
  ✓ banks: 18986 rows → banks.csv
  ✓ accounts: 423684 rows → accounts.csv
  ✓ transfers: 1000000 rows → transfers.csv
  ✓ bank_country: 18986 rows → bank_country.csv
  ✓ account_bank: 423684 rows → account_bank.csv
  ✓ alerts: 575 rows → alerts.csv
  ✓ account_country: 558170 rows → account_country.csv
  ✓ account_alert: 575 rows → account_alert.csv
[3/3] Generating Trino SQL: /tmp/ignored_seed_sql.sql

✅ Complete!
   SQL file: /tmp/ignored_seed_sql.sql
   CSV directory: /tmp/iceberg-csv-gen

Data summary:
   countries                  10 rows
   banks                   18986 rows
   a

Iceberg seeding complete. Transfers rows: **1000000**

## Quick Verify

Run this cell to verify backend health, Trino connectivity, and seeded Iceberg tables.


In [4]:
def quick_verify_iceberg() -> bool:
    checks = []

    try:
        health_resp = requests.get(f"{BASE_URL}/health", timeout=5)
        health_ok = False
        health_detail = ""

        if health_resp.ok:
            try:
                payload = health_resp.json()
                if isinstance(payload, dict):
                    health_ok = str(payload.get("status", "")).strip().upper() == "OK"
                    health_detail = _json.dumps(payload)
                else:
                    text_payload = str(payload).strip()
                    health_ok = text_payload.upper() == "OK"
                    health_detail = text_payload
            except ValueError:
                text_payload = (health_resp.text or "").strip()
                health_ok = text_payload.upper() == "OK"
                health_detail = text_payload
        else:
            health_detail = f"HTTP {health_resp.status_code}: {(health_resp.text or '').strip()}"

        checks.append(("Backend", health_ok, health_detail))
    except Exception as e:
        checks.append(("Backend", False, str(e)))

    trino_ping = run_trino("SELECT 1 AS ok")
    trino_ok = "error" not in trino_ping.columns
    trino_detail = "ok" if trino_ok else trino_ping.to_string(index=False)[:240]
    checks.append(("Trino", trino_ok, trino_detail))

    table_probe = run_trino("SELECT count(*) AS c FROM accounts")
    table_ok = "error" not in table_probe.columns
    table_count = None
    if table_ok and "c" in table_probe.columns and len(table_probe.index) > 0:
        table_count = table_probe.iloc[0]["c"]
    table_detail = f"accounts={table_count}" if table_ok else table_probe.to_string(index=False)[:240]
    checks.append(("Seed data", table_ok, table_detail))

    all_ok = all(ok for _, ok, _ in checks)
    summary = "PASS" if all_ok else "FAIL"
    display(Markdown(f"**Health summary:** `{summary}`"))

    for name, ok, detail in checks:
        status = "PASS" if ok else "FAIL"
        display(Markdown(f"- **{name}:** `{status}`"))
        if not ok and detail:
            display(Markdown(f"  - detail: `{detail}`"))

    if not all_ok:
        display(Markdown("**Action:** run `run_local_iceberg_setup(run_down=False, run_up=True)` then retry."))
    return all_ok


quick_verify_iceberg()


**Health summary:** `PASS`

- **Backend:** `PASS`

- **Trino:** `PASS`

- **Seed data:** `PASS`

True

## Query Sections

This notebook mirrors all query titles from `aml_demo_queries.ipynb` and runs each via Iceberg translation + Trino execution.


## Simple Queries


### S1 Count accounts

Count unique Account vertices.


In [6]:
gremlin = "g.V().hasLabel('Account').count()"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='S1 Count accounts', limit=10, tx_mode=False)


### S1 Count accounts

**Gremlin:**

```groovy
g.V().hasLabel('Account').count()
```

**Iceberg SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml.accounts
```

,count
0,423684


### S2 Count banks

Count Bank vertices - one per distinct bank ID in the data.


In [7]:
gremlin = "g.V().hasLabel('Bank').count()"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='S2 Count banks', limit=10, tx_mode=False)


### S2 Count banks

**Gremlin:**

```groovy
g.V().hasLabel('Bank').count()
```

**Iceberg SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml.banks
```

,count
0,18986


### S3 Count countries

Count Country vertices (10 pre-seeded jurisdictions).


In [8]:
gremlin = "g.V().hasLabel('Country').count()"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='S3 Count countries', limit=10, tx_mode=False)


### S3 Count countries

**Gremlin:**

```groovy
g.V().hasLabel('Country').count()
```

**Iceberg SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml.countries
```

,count
0,10


### S4 Count alerts

Count Alert vertices - one raised per suspicious transfer.


In [9]:
gremlin = "g.V().hasLabel('Alert').count()"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='S4 Count alerts', limit=10, tx_mode=False)


### S4 Count alerts

**Gremlin:**

```groovy
g.V().hasLabel('Alert').count()
```

**Iceberg SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml.alerts
```

,count
0,575


### S5 Count transfers

Count all TRANSFER edges.


In [10]:
gremlin = "g.E().hasLabel('TRANSFER').count()"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='S5 Count transfers', limit=10, tx_mode=False)


### S5 Count transfers

**Gremlin:**

```groovy
g.E().hasLabel('TRANSFER').count()
```

**Iceberg SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml.transfers
```

,count
0,1000000


### S6 Suspicious transfer count

Count confirmed suspicious TRANSFER edges (isLaundering=1).


In [11]:
gremlin = "g.E().has('isLaundering','1').count()"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='S6 Suspicious transfer count', limit=10, tx_mode=False)


### S6 Suspicious transfer count

**Gremlin:**

```groovy
g.E().has('isLaundering','1').count()
```

**Iceberg SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml.transfers WHERE is_laundering = ?
```

**Parameters:** `['1']`

,count
0,575


### S7 High-risk countries

List Country vertices with riskLevel=HIGH.


In [12]:
gremlin = "g.V().hasLabel('Country').has('riskLevel','HIGH').project('countryCode','countryName','region','fatfBlacklist').by('countryCode').by('countryName').by('region').by('fatfBlacklist')"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='S7 High-risk countries', limit=10, tx_mode=False)


### S7 High-risk countries

**Gremlin:**

```groovy
g.V().hasLabel('Country').has('riskLevel','HIGH').project('countryCode','countryName','region','fatfBlacklist').by('countryCode').by('countryName').by('region').by('fatfBlacklist')
```

**Iceberg SQL Translation:**

```sql
SELECT v.country_code AS "countryCode", v.country_name AS "countryName", v.region AS "region", v.fatf_blacklist AS "fatfBlacklist" FROM aml.countries v WHERE v.risk_level = ?
```

**Parameters:** `['HIGH']`

,countryCode,countryName,region,fatfBlacklist
0,NG,Nigeria,Africa,false
1,KY,Cayman Islands,Americas,true
2,PA,Panama,Americas,true


### S8 High-severity alerts

Show HIGH-severity open alerts.


In [13]:
gremlin = "g.V().hasLabel('Alert').has('severity','HIGH').project('alertId','alertType','status','raisedAt').by('alertId').by('alertType').by('status').by('raisedAt').limit(15)"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='S8 High-severity alerts', limit=15, tx_mode=False)


### S8 High-severity alerts

**Gremlin:**

```groovy
g.V().hasLabel('Alert').has('severity','HIGH').project('alertId','alertType','status','raisedAt').by('alertId').by('alertType').by('status').by('raisedAt').limit(15)
```

**Iceberg SQL Translation:**

```sql
SELECT v.alert_id AS "alertId", v.alert_type AS "alertType", v.status AS "status", v.raised_at AS "raisedAt" FROM aml.alerts v WHERE v.severity = ? LIMIT 15
```

**Parameters:** `['HIGH']`

,alertId,alertType,status,raisedAt
0,ALERT-4743,SUSPICIOUS_TRANSFER,OPEN,2022/09/01 00:21
1,ALERT-85764,SUSPICIOUS_TRANSFER,OPEN,2022/09/01 00:03
2,ALERT-137526,SUSPICIOUS_TRANSFER,OPEN,2022/09/01 09:56
3,ALERT-216901,SUSPICIOUS_TRANSFER,OPEN,2022/09/01 00:09
4,ALERT-299902,SUSPICIOUS_TRANSFER,OPEN,2022/09/01 00:20
5,ALERT-317687,SUSPICIOUS_TRANSFER,OPEN,2022/09/01 00:03
6,ALERT-317695,SUSPICIOUS_TRANSFER,OPEN,2022/09/03 10:20
7,ALERT-317696,SUSPICIOUS_TRANSFER,OPEN,2022/09/03 12:08
8,ALERT-317702,SUSPICIOUS_TRANSFER,OPEN,2022/09/04 03:24
9,ALERT-318130,SUSPICIOUS_TRANSFER,OPEN,2022/09/04 05:06


## Complex Queries


### C1 Top sender accounts

Accounts ranked by outgoing transfer count - find the biggest hubs.


In [14]:
gremlin = "g.V().hasLabel('Account').project('accountId','bankId','outDegree').by('accountId').by('bankId').by(outE('TRANSFER').count()).order().by(select('outDegree'),Order.desc).limit(15)"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='C1 Top sender accounts', limit=15, tx_mode=False)


### C1 Top sender accounts

**Gremlin:**

```groovy
g.V().hasLabel('Account').project('accountId','bankId','outDegree').by('accountId').by('bankId').by(outE('TRANSFER').count()).order().by(select('outDegree'),Order.desc).limit(15)
```

**Iceberg SQL Translation:**

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT COUNT(*) FROM aml.transfers WHERE out_id = v.id) AS "outDegree" FROM aml.accounts v ORDER BY outDegree DESC LIMIT 15
```

,accountId,bankId,outDegree
0,100428660,070,19968
1,1004286A8,070,11967
2,100428978,070,2441
3,1004286F0,070,2142
4,1004289C0,070,1993
5,100428810,070,1982
6,100428780,070,1968
7,1004287C8,070,1596
8,100428738,070,1578
9,100428A51,070,1522


### C2 Suspicious hubs

Accounts with the most suspicious outgoing transfers.


In [15]:
gremlin = "g.V().hasLabel('Account').project('accountId','bankId','suspiciousOut','totalOut').by('accountId').by('bankId').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).where(select('suspiciousOut').is(gt(0))).order().by(select('suspiciousOut'),Order.desc).limit(15)"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='C2 Suspicious hubs', limit=15, tx_mode=False)


### C2 Suspicious hubs

**Gremlin:**

```groovy
g.V().hasLabel('Account').project('accountId','bankId','suspiciousOut','totalOut').by('accountId').by('bankId').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).where(select('suspiciousOut').is(gt(0))).order().by(select('suspiciousOut'),Order.desc).limit(15)
```

**Iceberg SQL Translation:**

```sql
SELECT * FROM (SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT COUNT(*) FROM aml.transfers WHERE out_id = v.id AND is_laundering = '1') AS "suspiciousOut", (SELECT COUNT(*) FROM aml.transfers WHERE out_id = v.id) AS "totalOut" FROM aml.accounts v) p WHERE p."suspiciousOut" > 0 ORDER BY suspiciousOut DESC LIMIT 15
```

,accountId,bankId,suspiciousOut,totalOut
0,100428660,070,33,19968
1,1004286A8,070,19,11967
2,808338D40,0213952,16,26
3,800737690,021174,16,24
4,80F984C70,0135981,14,23
5,8095AF170,0225734,13,27
6,808E44B10,0211,13,26
7,80F1AF380,0116,12,26
8,8001BB380,010,10,48
9,1004286F0,070,8,2142


### C3 Account -> Bank (BELONGS_TO)

Show which bank each account belongs to via BELONGS_TO.


In [16]:
gremlin = "g.V().hasLabel('Account').limit(15).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='C3 Account -> Bank (BELONGS_TO)', limit=15, tx_mode=False)


### C3 Account -> Bank (BELONGS_TO)

**Gremlin:**

```groovy
g.V().hasLabel('Account').limit(15).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())
```

**Iceberg SQL Translation:**

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT ARRAY_JOIN(ARRAY_AGG(_njv0.bank_name), ',') FROM aml.account_bank _nje0 JOIN aml.banks _njv0 ON _njv0.id = _nje0.in_id WHERE _nje0.out_id = v.id) AS "bankName" FROM aml.accounts v LIMIT 15
```

,accountId,bankId,bankName
0,8000EBD30,010,Bank-010
1,8000EBAC0,001,Bank-001
2,8000EC1E0,001,Bank-001
3,8000EC280,012,Bank-012
4,8017BF800,002439,Bank-002439
5,8000EDEC0,001,Bank-001
6,80AEF5310,0211050,Bank-0211050
7,8000F4510,001,Bank-001
8,8011305D0,011813,Bank-011813
9,8000F4580,03208,Bank-03208


### C4 Bank -> Country (LOCATED_IN)

Show which country each bank is headquartered in.


In [17]:
gremlin = "g.V().hasLabel('Bank').limit(15).project('bankId','bankName','countryCode','countryName').by('bankId').by('bankName').by('countryCode').by(out('LOCATED_IN').values('countryName').fold())"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='C4 Bank -> Country (LOCATED_IN)', limit=15, tx_mode=False)


### C4 Bank -> Country (LOCATED_IN)

**Gremlin:**

```groovy
g.V().hasLabel('Bank').limit(15).project('bankId','bankName','countryCode','countryName').by('bankId').by('bankName').by('countryCode').by(out('LOCATED_IN').values('countryName').fold())
```

**Iceberg SQL Translation:**

```sql
SELECT v.bank_id AS "bankId", v.bank_name AS "bankName", v.country_code AS "countryCode", (SELECT ARRAY_JOIN(ARRAY_AGG(_njv0.country_name), ',') FROM aml.bank_country _nje0 JOIN aml.countries _njv0 ON _njv0.id = _nje0.in_id WHERE _nje0.out_id = v.id) AS "countryName" FROM aml.banks v LIMIT 15
```

,bankId,bankName,countryCode,countryName
0,010,Bank-010,SG,Singapore
1,03208,Bank-03208,CH,Switzerland
2,001,Bank-001,SG,Singapore
3,03209,Bank-03209,HK,Hong Kong
4,012,Bank-012,NG,Nigeria
5,002439,Bank-002439,AE,UAE
6,0211050,Bank-0211050,SG,Singapore
7,011813,Bank-011813,DE,Germany
8,0245335,Bank-0245335,KY,Cayman Islands
9,0036056,Bank-0036056,AE,UAE


### C5 Two-hop: Account -> Bank -> Country

Traverse Account->Bank->Country in two hops.


In [18]:
gremlin = "g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='C5 Two-hop: Account -> Bank -> Country', limit=10, tx_mode=False)


### C5 Two-hop: Account -> Bank -> Country

**Gremlin:**

```groovy
g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)
```

**Iceberg SQL Translation:**

```sql
SELECT v0.account_id AS accountId0, v1.bank_name AS bankName1, v2.country_name AS countryName2 FROM (SELECT * FROM aml.accounts LIMIT 1) v0 JOIN aml.account_bank e1 ON e1.out_id = v0.id JOIN aml.banks v1 ON v1.id = e1.in_id JOIN aml.bank_country e2 ON e2.out_id = v1.id JOIN aml.countries v2 ON v2.id = e2.in_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) LIMIT 10
```

,accountId0,bankName1,countryName2
0,8000EBD30,Bank-010,Singapore


### C6 Accounts sending to high-risk countries (SENT_VIA)

Find accounts routing money via FATF-blacklisted countries.


In [19]:
gremlin = "g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(20)"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='C6 Accounts sending to high-risk countries (SENT_VIA)', limit=20, tx_mode=False)


### C6 Accounts sending to high-risk countries (SENT_VIA)

**Gremlin:**

```groovy
g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(20)
```

**Iceberg SQL Translation:**

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", v.risk_score AS "riskScore" FROM aml.accounts v WHERE EXISTS (SELECT 1 FROM aml.account_country _we JOIN aml.countries _wv ON _wv.id = _we.in_id WHERE _we.out_id = v.id AND _wv.fatf_blacklist = ?) LIMIT 20
```

**Parameters:** `['true']`

,accountId,bankId,riskScore
0,8000F5030,012,0.1
1,8000F4FE0,001,0.1
2,812ED62E0,0245335,0.1
3,80012FD90,010,0.1
4,80012FE00,012,0.1
5,80012FE50,001,0.1
6,8005E18F0,01665,0.1
7,8005E24C0,01665,0.1
8,800131B10,012,0.1
9,8131A9A80,0011642,0.1


### C7 Flagged accounts with alert detail (FLAGGED_BY)

Show investigated accounts with linked Alert severity.


In [20]:
gremlin = "g.V().hasLabel('Account').where(outE('FLAGGED_BY')).project('accountId','bankId','alertCount','highSeverity').by('accountId').by('bankId').by(outE('FLAGGED_BY').count()).by(out('FLAGGED_BY').has('severity','HIGH').count()).limit(20)"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='C7 Flagged accounts with alert detail (FLAGGED_BY)', limit=20, tx_mode=False)


### C7 Flagged accounts with alert detail (FLAGGED_BY)

**Gremlin:**

```groovy
g.V().hasLabel('Account').where(outE('FLAGGED_BY')).project('accountId','bankId','alertCount','highSeverity').by('accountId').by('bankId').by(outE('FLAGGED_BY').count()).by(out('FLAGGED_BY').has('severity','HIGH').count()).limit(20)
```

**Iceberg SQL Translation:**

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT COUNT(*) FROM aml.account_alert WHERE out_id = v.id) AS "alertCount", (SELECT COUNT(*) FROM aml.account_alert _e JOIN aml.alerts _tv ON _tv.id = _e.in_id WHERE _e.out_id = v.id AND _tv.severity = 'HIGH') AS "highSeverity" FROM aml.accounts v WHERE EXISTS (SELECT 1 FROM aml.account_alert we WHERE we.out_id = v.id) LIMIT 20
```

,accountId,bankId,alertCount,highSeverity
0,801009860,01674,1,0
1,800054B70,012,6,2
2,800056160,001,1,0
3,100428660,070,33,4
4,800117590,012,1,0
5,800119810,010,1,0
6,800137450,012,1,0
7,8001A6140,010,1,0
8,8001BB380,010,10,0
9,8002B9DE0,01420,1,0


### C8 Cross-bank suspicious flow

Suspicious transfers that cross bank boundaries.


In [21]:
gremlin = "g.E().has('isLaundering','1').project('fromBank','fromAcct','toBank','toAcct','amount','currency').by(outV().values('bankId')).by(outV().values('accountId')).by(inV().values('bankId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='C8 Cross-bank suspicious flow', limit=15, tx_mode=False)


### C8 Cross-bank suspicious flow

**Gremlin:**

```groovy
g.E().has('isLaundering','1').project('fromBank','fromAcct','toBank','toAcct','amount','currency').by(outV().values('bankId')).by(outV().values('accountId')).by(inV().values('bankId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)
```

**Iceberg SQL Translation:**

```sql
SELECT ov.bank_id AS "fromBank", ov.account_id AS "fromAcct", iv.bank_id AS "toBank", iv.account_id AS "toAcct", e.amount AS "amount", e.currency AS "currency" FROM aml.transfers e JOIN aml.accounts ov ON ov.id = e.out_id JOIN aml.accounts iv ON iv.id = e.in_id WHERE e.is_laundering = ? LIMIT 15
```

**Parameters:** `['1']`

,fromBank,fromAcct,toBank,toAcct,amount,currency
0,070,1004286F0,009571,8056D2690,1353.70,Yuan
1,0112078,808DE9400,019477,808DEA500,14260.32,Yuan
2,070,100428660,001124,800825340,389769.39,US Dollar
3,070,100428930,0115,80CF5B3C0,118.71,Brazil Real
4,006179,805B5BC60,008771,805B5C500,601.81,US Dollar
5,070,100428660,023537,806A3C0D0,1303.29,US Dollar
6,0012803,80C987C00,028800,80C988190,1657.88,US Dollar
7,0239173,80FF26B70,0130425,80FF26D00,1064.92,US Dollar
8,001655,802078880,005365,80207A270,3527.12,Euro
9,070,100428780,0112989,8052436B0,1277.41,Rupee


### C9 Three-hop money trail

Follow suspicious TRANSFER hops 3 steps deep.


In [22]:
gremlin = "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='C9 Three-hop money trail', limit=10, tx_mode=False)


### C9 Three-hop money trail

**Gremlin:**

```groovy
g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)
```

**Iceberg SQL Translation:**

```sql
SELECT v0.account_id AS accountId0, v1.account_id AS accountId1, v2.account_id AS accountId2, v3.account_id AS accountId3 FROM (SELECT * FROM aml.accounts WHERE EXISTS (SELECT 1 FROM aml.transfers we WHERE we.out_id = id AND we.is_laundering = ?) LIMIT 1) v0 JOIN aml.transfers e1 ON e1.out_id = v0.id JOIN aml.accounts v1 ON v1.id = e1.in_id JOIN aml.transfers e2 ON e2.out_id = v1.id JOIN aml.accounts v2 ON v2.id = e2.in_id JOIN aml.transfers e3 ON e3.out_id = v2.id JOIN aml.accounts v3 ON v3.id = e3.in_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) AND v3.id NOT IN (v0.id, v1.id, v2.id) LIMIT 10
```

**Parameters:** `['1']`

,result
0,empty


### C10 Five-hop layering chain

Extended 5-hop traversal for layering detection.


In [23]:
gremlin = "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(5).path().by('accountId').limit(10)"
result = run_iceberg_query(gremlin, tx_mode=False)
display_iceberg_result(gremlin, result, title='C10 Five-hop layering chain', limit=10, tx_mode=False)


### C10 Five-hop layering chain

**Gremlin:**

```groovy
g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(5).path().by('accountId').limit(10)
```

**Iceberg SQL Translation:**

```sql
SELECT v0.account_id AS accountId0, v1.account_id AS accountId1, v2.account_id AS accountId2, v3.account_id AS accountId3, v4.account_id AS accountId4, v5.account_id AS accountId5 FROM (SELECT * FROM aml.accounts WHERE EXISTS (SELECT 1 FROM aml.transfers we WHERE we.out_id = id AND we.is_laundering = ?) LIMIT 1) v0 JOIN aml.transfers e1 ON e1.out_id = v0.id JOIN aml.accounts v1 ON v1.id = e1.in_id JOIN aml.transfers e2 ON e2.out_id = v1.id JOIN aml.accounts v2 ON v2.id = e2.in_id JOIN aml.transfers e3 ON e3.out_id = v2.id JOIN aml.accounts v3 ON v3.id = e3.in_id JOIN aml.transfers e4 ON e4.out_id = v3.id JOIN aml.accounts v4 ON v4.id = e4.in_id JOIN aml.transfers e5 ON e5.out_id = v4.id JOIN aml.accounts v5 ON v5.id = e5.in_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) AND v3.id NOT IN (v0.id, v1.id, v2.id) AND v4.id NOT IN (v0.id, v1.id, v2.id, v3.id) AND v5.id NOT IN (v0.id, v1.id, v2.id, v3.id, v4.id) LIMIT 10
```

**Parameters:** `['1']`

,result
0,empty


### C11 Transactional suspicious count

Suspicious transfer count via transactional endpoint.


In [24]:
gremlin = "g.E().has('isLaundering','1').count()"
result = run_iceberg_query(gremlin, tx_mode=True)
display_iceberg_result(gremlin, result, title='C11 Transactional suspicious count', limit=10, tx_mode=True)


### C11 Transactional suspicious count

**Gremlin:**

```groovy
g.E().has('isLaundering','1').count()
```

**Iceberg SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml.transfers WHERE is_laundering = ?
```

**Parameters:** `['1']`

,count
0,575


## Done
